In [13]:
"""
Batch inference utility for the `scrfd` pip package.

- Recursively traverse an image directory (or run on a single image).
- Run face detection with a configurable probability threshold.
- Save per-image label files in the format: "score x1 y1 x2 y2" (pixels).
- Optionally save images with bounding boxes drawn.
- Auto-increment output runs: runs/scrfd/exp, exp2, exp3, ...

Usage (as a script):
    python scrfd_batch.py \
        --weights ./models/scrfd.onnx \
        --source ./images \
        --project runs/scrfd \
        --name exp \
        --prob-thres 0.4 \
        --save-img --save-txt

Usage (in a notebook):
    from scrfd_batch import run_scrfd_folder
    run_scrfd_folder(
        weights_path="./models/scrfd.onnx",
        source="./images",
        project="runs/scrfd",
        name="exp",
        prob_thres=0.4,
        save_img=True,
        save_txt=True,
    )
"""

#from __future__ import annotations

import argparse
import itertools
from pathlib import Path
from typing import Iterable, List, Sequence, Tuple, Union

from PIL import Image, ImageDraw, ImageFont
from scrfd import SCRFD, Threshold

# ----------------------------- Constants -----------------------------

IMG_EXTS: Tuple[str, ...] = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")


# ----------------------------- Utilities -----------------------------

def list_images_recursive(source: str | Path) -> List[Path]:
    """Return a sorted list of image paths found in `source` (file or directory, recursive)."""
    p = Path(source)
    if p.is_file() and p.suffix.lower() in IMG_EXTS:
        return [p]
    if p.is_dir():
        return sorted(q for q in p.rglob("*") if q.suffix.lower() in IMG_EXTS)
    raise FileNotFoundError(f"Source not found or unsupported: {source}")


def increment_path(base: Path, exist_ok: bool = False) -> Path:
    """Return a unique output path by auto-incrementing a trailing integer suffix (exp2, exp3, ...)."""
    if exist_ok or not base.exists():
        return base
    for i in itertools.count(2):
        cand = base.parent / f"{base.name}{i}"
        if not cand.exists():
            return cand


def _draw_text_with_bg(draw: ImageDraw.ImageDraw, xy: Tuple[int, int], text: str) -> None:
    """Draw a black bg rectangle then white text; improves readability."""
    font = ImageFont.load_default()
    x, y = xy
    bbox = draw.textbbox((x, y), text, font=font)
    pad = 2
    bg = (bbox[0] - pad, bbox[1] - pad, bbox[2] + pad, bbox[3] + pad)
    draw.rectangle(bg, fill=(0, 0, 0))
    draw.text((x, y), text, fill=(255, 255, 255), font=font)


def draw_box_and_label(pil_img: Image.Image, box_xyxy: Sequence[int | float], score: float) -> None:
    """Draw a bounding box and score label on a PIL image (in-place)."""
    draw = ImageDraw.Draw(pil_img)
    x1, y1, x2, y2 = (int(round(v)) for v in box_xyxy)
    thickness = max(2, int(0.002 * (pil_img.width + pil_img.height)))
    for offs in range(thickness):
        draw.rectangle([(x1 - offs, y1 - offs), (x2 + offs, y2 + offs)], outline=(0, 255, 0), width=1)
    _draw_text_with_bg(draw, (x1, max(0, y1 - 14)), f"{score:.3f}")


# ----------------------------- BBox Normalizer -----------------------------
def normalize_bbox_to_xyxy(bbox: Union[Sequence, object]) -> Tuple[float, float, float, float]:
    """
    Normalize different bbox representations to (x1, y1, x2, y2) as floats.

    Supports:
    - flat iterable of 4 numbers: [x1, y1, x2, y2]
    - iterable of 2 points: [(x1, y1), (x2, y2)]
    - dict-like with keys: ('x1','y1','x2','y2') OR ('left','top','right','bottom') OR ('x','y','w','h')
    - object with attributes: .x1/.y1/.x2/.y2 OR .left/.top/.right/.bottom
    - object with .left_top/.right_bottom (each a 2-tuple)
    - **scrfd.schemas.Bbox**: .upper_left (Point{x,y}) and .lower_right (Point{x,y})
    - object with (x, y, w, h) attributes

    Raises:
        ValueError if the format cannot be interpreted.
    """
    # 1) Sequence types
    if isinstance(bbox, (list, tuple)):
        # a) flat [x1, y1, x2, y2]
        if len(bbox) == 4 and all(isinstance(v, (int, float)) for v in bbox):
            x1, y1, x2, y2 = bbox
            return float(x1), float(y1), float(x2), float(y2)
        # b) [(x1, y1), (x2, y2)]
        if len(bbox) == 2 and all(isinstance(pt, (list, tuple)) and len(pt) == 2 for pt in bbox):
            (x1, y1), (x2, y2) = bbox
            return float(x1), float(y1), float(x2), float(y2)

    # 2) Dict-like
    if isinstance(bbox, dict):
        keys = set(bbox.keys())
        if {'x1', 'y1', 'x2', 'y2'} <= keys:
            return float(bbox['x1']), float(bbox['y1']), float(bbox['x2']), float(bbox['y2'])
        if {'left', 'top', 'right', 'bottom'} <= keys:
            return float(bbox['left']), float(bbox['top']), float(bbox['right']), float(bbox['bottom'])
        if {'x', 'y', 'w', 'h'} <= keys:
            x, y, w, h = bbox['x'], bbox['y'], bbox['w'], bbox['h']
            return float(x), float(y), float(x) + float(w), float(y) + float(h)

    # 3) Object with direct xyxy attrs
    for attrs in (('x1', 'y1', 'x2', 'y2'), ('left', 'top', 'right', 'bottom')):
        if all(hasattr(bbox, a) for a in attrs):
            x1, y1, x2, y2 = (getattr(bbox, a) for a in attrs)
            return float(x1), float(y1), float(x2), float(y2)

    # 4) Object with .left_top and .right_bottom
    if all(hasattr(bbox, a) for a in ('left_top', 'right_bottom')):
        lt, rb = getattr(bbox, 'left_top'), getattr(bbox, 'right_bottom')
        if isinstance(lt, (list, tuple)) and isinstance(rb, (list, tuple)) and len(lt) == len(rb) == 2:
            x1, y1 = lt
            x2, y2 = rb
            return float(x1), float(y1), float(x2), float(y2)
        # handle point-like objects with .x/.y
        if hasattr(lt, 'x') and hasattr(lt, 'y') and hasattr(rb, 'x') and hasattr(rb, 'y'):
            return float(lt.x), float(lt.y), float(rb.x), float(rb.y)

    # 5) **scrfd.schemas.Bbox**: .upper_left and .lower_right (Point{x,y})
    if all(hasattr(bbox, a) for a in ('upper_left', 'lower_right')):
        ul, lr = getattr(bbox, 'upper_left'), getattr(bbox, 'lower_right')
        # Points may be tuple-like or attr-based
        if hasattr(ul, 'x') and hasattr(ul, 'y') and hasattr(lr, 'x') and hasattr(lr, 'y'):
            return float(ul.x), float(ul.y), float(lr.x), float(lr.y)
        if isinstance(ul, (list, tuple)) and isinstance(lr, (list, tuple)) and len(ul) == len(lr) == 2:
            (x1, y1), (x2, y2) = ul, lr
            return float(x1), float(y1), float(x2), float(y2)

    # 6) Object with (x, y, w, h)
    if all(hasattr(bbox, a) for a in ('x', 'y', 'w', 'h')):
        x, y, w, h = bbox.x, bbox.y, bbox.w, bbox.h
        return float(x), float(y), float(x) + float(w), float(y) + float(h)

    raise ValueError(f"Unrecognized bbox format: {type(bbox)} -> {bbox}")


# ----------------------------- Core Runner -----------------------------

def run_scrfd_folder(
    weights_path: str | Path,
    source: str | Path,
    project: str | Path = "runs/scrfd",
    name: str = "exp",
    exist_ok: bool = False,
    prob_thres: float = 0.4,
    save_img: bool = True,
    save_txt: bool = True,
    save_empty: bool = False,
) -> None:
    """
    Run SCRFD detection over a folder (recursive) or a single image.

    For each image:
      - Save a label file at {save_dir}/labels/{stem}.txt with lines:
            "score x1 y1 x2 y2"
        (pixel coordinates in the original image)
      - Optionally save an annotated image with bounding boxes at {save_dir}/{filename}
    """
    save_dir = increment_path(Path(project) / name, exist_ok=exist_ok)
    save_dir.mkdir(parents=True, exist_ok=True)
    labels_dir = save_dir / "labels"
    if save_txt:
        labels_dir.mkdir(parents=True, exist_ok=True)

    face_detector = SCRFD.from_path(str(weights_path))
    threshold = Threshold(probability=prob_thres)

    paths = list_images_recursive(source)
    if not paths:
        print(f"[WARN] No images found in: {source}")
        return

    print(f"Found {len(paths)} images. Output -> {save_dir}")

    for i, img_path in enumerate(paths, 1):
        img = Image.open(img_path).convert("RGB")
        faces = face_detector.detect(img, threshold=threshold)

        out_img_path = save_dir / img_path.name
        label_path = labels_dir / f"{img_path.stem}.txt"

        lines: List[str] = []
        for face in faces:
            # Robust bbox handling (flat 4-tuple OR 2 points)
            x1, y1, x2, y2 = normalize_bbox_to_xyxy(face.bbox)
            score = float(face.probability)

            if save_img:
                draw_box_and_label(img, (x1, y1, x2, y2), score)
            if save_txt:
                xi1, yi1, xi2, yi2 = (int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2)))
                lines.append(f"{score:.6f} {xi1} {yi1} {xi2} {yi2}")

        if save_txt:
            if lines:
                with open(label_path, "w") as f:
                    f.write("\n".join(lines))
            elif save_empty:
                open(label_path, "w").close()

        if save_img:
            img.save(out_img_path)

        if (i % 50 == 0) or (i == len(paths)):
            print(f"[{i}/{len(paths)}] {img_path.name} - dets={len(faces)}")

    print("Done.")


In [14]:
run_scrfd_folder(
    weights_path="/home/jocareher/Downloads/scrfd.onnx",
    source="/home/jocareher/Downloads/obbabyface_rot_adults_vs_children/test/images",
    project="/home/jocareher/Downloads/scrfd",
    name="exp",
    exist_ok=True,
    prob_thres=0.1,
    save_img=True,
    save_txt=True,
    save_empty=False,
)

Found 2172 images. Output -> /home/jocareher/Downloads/scrfd/exp
[50/2172] 29c504b55c047d6c.jpg - dets=1
[100/2172] 59b80f5524305da8.jpg - dets=7
[150/2172] 8ed11dd987c3df64.jpg - dets=2
[200/2172] cb7d9623640a3c86.jpg - dets=1
[250/2172] face_bcn_131.jpg - dets=4
[300/2172] face_bcn_68.jpg - dets=1
[350/2172] face_img_1550.jpg - dets=3
[400/2172] face_img_2169.jpg - dets=4
[450/2172] face_img_2600.jpg - dets=2
[500/2172] face_img_3038.jpg - dets=3
[550/2172] face_img_3503.jpg - dets=2
[600/2172] face_img_3947.jpg - dets=2
[650/2172] face_img_449.jpg - dets=4
[700/2172] face_img_4912.jpg - dets=3
[750/2172] face_img_5418.jpg - dets=2
[800/2172] face_img_5895.jpg - dets=1
[850/2172] face_img_6366.jpg - dets=7
[900/2172] face_img_6747.jpg - dets=3
[950/2172] face_img_7207.jpg - dets=2
[1000/2172] face_img_7665.jpg - dets=6
[1050/2172] face_img_8068.jpg - dets=6
[1100/2172] face_img_8529.jpg - dets=3
[1150/2172] flip_face_bcn_11.jpg - dets=5
[1200/2172] flip_face_bcn_652.jpg - dets=4
[125

In [ ]:
# # ----------------------------- CLI Entrypoint -----------------------------

# def _build_argparser() -> argparse.ArgumentParser:
#     """
#     Build the command-line argument parser.

#     Returns
#     -------
#     argparse.ArgumentParser
#         Configured argument parser for the script.
#     """
#     ap = argparse.ArgumentParser(
#         description="Batch inference with the `scrfd` pip package. "
#                     "Saves labels as 'score x1 y1 x2 y2' and (optionally) annotated images."
#     )
#     ap.add_argument("--weights", type=str, required=True, help="Path to SCRFD ONNX model (e.g., ./models/scrfd.onnx)")
#     ap.add_argument("--source", type=str, required=True, help="Path to an image or a directory of images (recursive).")
#     ap.add_argument("--project", type=str, default="runs/scrfd", help="Base output directory.")
#     ap.add_argument("--name", type=str, default="exp", help="Run subfolder name under the project directory.")
#     ap.add_argument("--exist-ok", action="store_true", help="Reuse existing run folder if it exists.")
#     ap.add_argument("--prob-thres", type=float, default=0.4, help="Probability threshold for detections.")
#     ap.add_argument("--save-img", action="store_true", help="Save annotated images to the run folder.")
#     ap.add_argument("--save-txt", action="store_true", help="Save labels to run/labels/*.txt.")
#     ap.add_argument("--save-empty", action="store_true", help="Write empty .txt files for images with no detections.")
#     return ap


# def _main_cli() -> None:
#     """
#     Command-line entrypoint. Parses arguments and runs batch inference.
#     """
#     ap = _build_argparser()
#     args = ap.parse_args()

#     run_scrfd_folder(
#         weights_path=args.weights,
#         source=args.source,
#         project=args.project,
#         name=args.name,
#         exist_ok=args.exist_ok,
#         prob_thres=args.prob_thres,
#         save_img=args.save_img,
#         save_txt=args.save_txt,
#         save_empty=args.save_empty,
#     )


# if __name__ == "__main__":
#     _main_cli()